In [2]:
import pandas as pd

train = pd.read_parquet("../data_processed/train.parquet")
val = pd.read_parquet("../data_processed/val.parquet")
test = pd.read_parquet("../data_processed/test.parquet")
future_2026 = pd.read_parquet("../data_processed/future_2026.parquet")

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)
print("2026:", future_2026.shape)

Train: (24200, 42)
Val: (1200, 42)
Test: (1838, 42)
2026: (198, 42)


In [3]:
print(train.columns.tolist())

['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name', 'date', 'driverRef', 'forename', 'surname', 'nationality', 'constructorRef', 'name_constructor', 'nationality_constructor', 'status', 'quali_position', 'grid_fixed', 'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3', 'feat_circuit_history', 'dnf', 'feat_driver_dnf_rate_last5', 'prev_points', 'prev_standing_position', 'podium']


In [4]:
for split_df in [train, val, test, future_2026]:
    split_df['podium'] = (split_df['positionOrder'] <= 3).astype(int)

print(train['podium'].mean(), val['podium'].mean(), test['podium'].mean())

0.12450413223140495 0.15 0.1501632208922742


In [5]:
train.to_parquet("../data_processed/train.parquet", index=False)
val.to_parquet("../data_processed/val.parquet", index=False)
test.to_parquet("../data_processed/test.parquet", index=False)
future_2026.to_parquet("../data_processed/future_2026.parquet", index=False)
print("Re-saved with podium column included.")

Re-saved with podium column included.


In [6]:
feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

X_train, y_train = train[feature_cols], train['podium']
X_val, y_val = val[feature_cols], val['podium']
X_test, y_test = test[feature_cols], test['podium']

print(y_train.mean(), y_val.mean(), y_test.mean())

0.12450413223140495 0.15 0.1501632208922742


In [7]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handles class imbalance
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [9]:
from sklearn.metrics import roc_auc_score, log_loss, classification_report

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = model.predict(X_val)

print("AUC-ROC:", roc_auc_score(y_val, val_probs))
print("Log Loss:", log_loss(y_val, val_probs))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

AUC-ROC: 0.9307244008714597
Log Loss: 0.3870920240879059

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.81      0.89      1020
           1       0.46      0.92      0.61       180

    accuracy                           0.83      1200
   macro avg       0.72      0.86      0.75      1200
weighted avg       0.90      0.83      0.85      1200



In [10]:
val_eval = val.copy()
val_eval['pred_prob'] = val_probs

def top3_accuracy(df_eval):
    correct = 0
    total = 0
    for race_id, group in df_eval.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        predicted_podium = set(group.nlargest(3, 'pred_prob')['driverId'])
        correct += len(actual_podium & predicted_podium)
        total += len(actual_podium)
    return correct / total

print("Top-3 match rate:", top3_accuracy(val_eval))

Top-3 match rate: 0.7


In [11]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

feat_grid                     0.751418
prev_standing_position        0.112087
feat_team_form_last3          0.034976
prev_points                   0.032071
feat_driver_dnf_rate_last5    0.028552
feat_driver_form_last3        0.020866
feat_circuit_history          0.020030
dtype: float32
